# Test GPU

In [ ]:
!nvidia-smi

# Instalacja bibliotek

In [ ]:
pip install transformers accelerate qwen-vl-utils pillow

In [ ]:
pip install torch --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import json
import time
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# Konfiguracja

In [ ]:
FOLDER_DOKUMENTOW = "./dokumenty_testowe"
FOLDER_WYNIKOW = "./output"
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# Te same pola co w qwen.ipynb (SCHEMA_POL) - tu szukamy ich lokalizacji na obrazie,
# a nie samej wartosci.
NAZWY_POL = [
    "nazwa_wierzyciela",
    "krs",
    "nip",
    "regon",
    "adres_wierzyciela",
    "waluta",
    "wysokosc_wierzytelnosci",
    "termin_zaplaty",
    "sporna",
    "zakres_podstawa_sporu",
    "odsetki",
    "odsetki_do",
    "uwagi",
]

INSTRUKCJA_GROUNDING = f"""Przeanalizuj ten dokument sadowy dotyczacy wierzytelnosci.
Dla KAZDEGO z ponizszych pol, jesli faktycznie wystepuje w dokumencie, znajdz dokladne miejsce
na obrazie, w ktorym ta informacja jest zapisana (np. fragment tekstu, etykiete wraz z wartoscia,
wpis w tabeli).

Pola do znalezienia: {", ".join(NAZWY_POL)}

KRYTYCZNE ZASADY - nie zgaduj, nie przyblizaj, nie forsuj dopasowan na sile:
- Pole przypisuj TYLKO wtedy, gdy w dokumencie faktycznie wystepuje ta konkretna informacja,
  jednoznacznie z nia zwiazana (np. widoczna etykieta, naglowek kolumny, jasny kontekst zdania).
- krs / nip / regon: przypisuj TYLKO gdy przy numerze widnieje wprost skrot "KRS", "NIP" lub
  "REGON". NIE myl tych numerow z PESEL, numerem sprawy, numerem tytulu wykonawczego ani zadnym
  innym identyfikatorem - jesli dany numer nie ma przy sobie takiej etykiety, pomin to pole
  calkowicie zamiast zwracac "najbardziej podobny" numer.
- nazwa_wierzyciela / adres_wierzyciela: pobieraj WYLACZNIE z sekcji dotyczacej wierzyciela
  (np. "Oznaczenie wierzyciela", "Wierzyciel:", adresat bedacy wierzycielem). NIE pobieraj danych
  z sekcji dotyczacej dluznika / zobowiazanego (np. "Oznaczenie zobowiazanego", "Dluznik:") - to
  inny podmiot, nawet jesli dane wygladaja podobnie ulozone.
- Jesli nie masz pewnosci, czy dany fragment faktycznie odpowiada danemu polu, POMIN to pole
  zamiast zwracac niepewna lub przyblizona lokalizacje.

Zwroc WYLACZNIE liste JSON (bez dodatkowego tekstu, bez markdown code-fence), gdzie kazdy element to:
{{
  "pole": "nazwa_pola_z_listy_powyzej",
  "wartosc": "odczytana wartosc tekstowa",
  "bbox_2d": [x1, y1, x2, y2]
}}

bbox_2d to wspolrzedne prostokata w pikselach (lewy-gorny róg x1,y1 oraz prawy-dolny róg x2,y2)
dokladnie otaczajacego miejsce na obrazie, gdzie ta wartosc jest napisana.

Jesli pole nie wystepuje w dokumencie albo nie da sie go jednoznacznie zlokalizowac, pomin je
calkowicie (nie dodawaj wpisu do listy)."""

# Kolory dla poszczegolnych pol (cyklicznie, jesli pol wiecej niz kolorow)
PALETA_KOLOROW = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#46f0f0", "#f032e6", "#bcf60c", "#fabebe", "#008080",
    "#e6beff", "#9a6324", "#800000",
]
MAPA_KOLOROW = {nazwa: PALETA_KOLOROW[i % len(PALETA_KOLOROW)] for i, nazwa in enumerate(NAZWY_POL)}

# Model

In [ ]:
def zaladuj_model():
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    return model, processor


def uruchom_grounding(model, processor, sciezka_obrazu: str) -> dict:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sciezka_obrazu},
                {"type": "text", "text": INSTRUKCJA_GROUNDING}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
    ).to("cuda")

    czas_start = time.time()
    output_ids = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
    czas_trwania = time.time() - czas_start

    output_text = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )[0]

    # WAZNE: Qwen2.5-VL przed przetworzeniem przeskalowuje obraz do wlasnej rozdzielczosci
    # (smart_resize). Zwracane wspolrzedne bbox_2d odnosza sie do TEGO przeskalowanego obrazu,
    # a nie do oryginalnego pliku. image_inputs[0] to dokladnie ten obraz "widziany" przez model,
    # wiec rysujemy ramki na nim, zeby uniknac bledow przeliczania skali.
    obraz_widziany_przez_model = image_inputs[0]

    return {
        "surowy_output": output_text,
        "czas_s": round(czas_trwania, 2),
        "obraz": obraz_widziany_przez_model,
    }

# Parsowanie odpowiedzi i rysowanie ramek

In [ ]:
def sprobuj_sparsowac_json_liste(tekst: str):
    tekst = tekst.strip()
    if tekst.startswith("```"):
        tekst = tekst.strip("`")
        if tekst.startswith("json"):
            tekst = tekst[4:]
    start = tekst.find("[")
    end = tekst.rfind("]")
    if start == -1 or end == -1:
        return None, "brak_listy_json_w_odpowiedzi"
    try:
        return json.loads(tekst[start:end + 1]), None
    except json.JSONDecodeError as e:
        return None, f"blad_parsowania: {e}"


def narysuj_zaznaczenia(obraz: Image.Image, wpisy: list) -> Image.Image:
    obraz = obraz.convert("RGB").copy()
    draw = ImageDraw.Draw(obraz)

    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", 16)
    except OSError:
        font = ImageFont.load_default()

    for wpis in wpisy:
        pole = wpis.get("pole", "?")
        bbox = wpis.get("bbox_2d")
        if not bbox or len(bbox) != 4:
            continue

        x1, y1, x2, y2 = bbox
        kolor = MAPA_KOLOROW.get(pole, "#ffffff")

        draw.rectangle([x1, y1, x2, y2], outline=kolor, width=3)

        podpis = pole
        bbox_tekstu = draw.textbbox((0, 0), podpis, font=font)
        szer_tekstu = bbox_tekstu[2] - bbox_tekstu[0]
        wys_tekstu = bbox_tekstu[3] - bbox_tekstu[1]

        pozycja_y = max(0, y1 - wys_tekstu - 6)
        draw.rectangle(
            [x1, pozycja_y, x1 + szer_tekstu + 8, pozycja_y + wys_tekstu + 6],
            fill=kolor
        )
        draw.text((x1 + 4, pozycja_y + 2), podpis, fill="#000000", font=font)

    return obraz

# MAIN
## Czyszczenie pamieci

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def main():
    folder_wej = Path(FOLDER_DOKUMENTOW)
    folder_wyj = Path(FOLDER_WYNIKOW)
    folder_wyj.mkdir(parents=True, exist_ok=True)

    pliki = sorted([p for p in folder_wej.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")])

    if not pliki:
        print(f"Brak plikow .jpg/.png w {FOLDER_DOKUMENTOW} - wrzuc tam dokumenty testowe.")
        return

    print("Ladowanie modelu Qwen2.5-VL-7B...")
    model, processor = zaladuj_model()
    print("Model zaladowany.\n")

    for plik in pliki:
        print(f"Przetwarzanie: {plik.name}")
        wynik = uruchom_grounding(model, processor, str(plik))
        wpisy, blad = sprobuj_sparsowac_json_liste(wynik["surowy_output"])

        if wpisy is None:
            print(f"  Blad parsowania: {blad}")
            print(f"  Surowy output: {wynik['surowy_output'][:300]}")
            continue

        obraz_zaznaczony = narysuj_zaznaczenia(wynik["obraz"], wpisy)

        nazwa_bazowa = plik.stem
        sciezka_obrazu_wyj = folder_wyj / f"{nazwa_bazowa}_zaznaczone.png"
        sciezka_json_wyj = folder_wyj / f"{nazwa_bazowa}_dane.json"

        obraz_zaznaczony.save(sciezka_obrazu_wyj)
        with open(sciezka_json_wyj, "w", encoding="utf-8") as f:
            json.dump({
                "plik_zrodlowy": plik.name,
                "czas_s": wynik["czas_s"],
                "znalezione_pola": wpisy,
            }, f, ensure_ascii=False, indent=2)

        print(f"  Znaleziono {len(wpisy)} pol -> {sciezka_obrazu_wyj.name}")

    print(f"\nGotowe. Zaznaczone dokumenty zapisane w folderze: {FOLDER_WYNIKOW}/")


if __name__ == "__main__":
    main()